# Claude Haiku churn pilot

This notebook first runs an 8-row smoke test from a reproducible 200-row test-set sample. Review formatting, latency, and failures before setting `RUN_FULL_BATCH = True` for additional billable API calls. Customer IDs, labels, and baseline outputs are retained locally and are never included in prompts.

## Configuration and credentials

The setup cell loads `ANTHROPIC_API_KEY` from the local, gitignored `.env` file and initializes the Anthropic client without printing the secret. Model and sample controls are listed next; the LLM receives profile attributes only, while IDs and labels remain available locally for evaluation.

In [6]:
from datetime import datetime, timezone
import json
import os
import random
import re
import time
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from IPython.display import display

working_dir = Path.cwd()
project_root = working_dir.parent if working_dir.name == "notebooks" else working_dir
env_path = project_root / ".env"
if not env_path.exists():
    print("No project .env file found. Create it from .env.example and add ANTHROPIC_API_KEY locally; never commit the key.")
load_dotenv(env_path, override=False)
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is missing. Add it to the local .env file (do not paste it into the notebook).")

client = anthropic.Anthropic(api_key=api_key, timeout=60.0)
MODEL_NAME = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")
print(f"Anthropic client initialized; configured model: {MODEL_NAME}")

Anthropic client initialized; configured model: claude-haiku-4-5-20251001


In [7]:
# Experiment controls: the full batch stays opt-in until the pilot has been reviewed.
DRY_RUN_SIZE = 8
MAIN_SAMPLE_SIZE = 200
RANDOM_SEED = 42
MAX_RETRIES = 4
INITIAL_BACKOFF_SECONDS = 1.0
REQUEST_PAUSE_SECONDS = 0.15
RUN_FULL_BATCH = False
RUN_SONNET_SIDE_EXPERIMENT = False
SONNET_MODEL_NAME = "claude-sonnet-4-5-20250929"
SONNET_SAMPLE_SIZE = 25

random.seed(RANDOM_SEED)
print(f"Planned main sample: {MAIN_SAMPLE_SIZE} test rows; smoke test first: {DRY_RUN_SIZE} rows.")
print("No full-batch or Sonnet requests are sent unless their run toggles are explicitly enabled.")

Planned main sample: 200 test rows; smoke test first: 8 rows.
No full-batch or Sonnet requests are sent unless their run toggles are explicitly enabled.


## Define the experiment before calling the API

The fixed seed makes the 200-row sample and its 8-row smoke subset reproducible. Retry and pause settings limit transient failures and request pacing. Keep both run toggles off until the small live check has valid, reviewable responses; API usage is billable.

In [8]:
source_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
baseline_path = project_root / "data" / "processed" / "baseline_predictions.csv"
source_df = pd.read_csv(source_path)
source_df["TotalCharges"] = pd.to_numeric(source_df["TotalCharges"], errors="coerce")
baseline_df = pd.read_csv(baseline_path)

# Join target/baseline evaluation fields locally; only profile features will enter prompts.
experiment_df = baseline_df.merge(
    source_df,
    on="customerID",
    how="left",
    validate="one_to_one",
)
if experiment_df["Churn"].isna().any():
    raise ValueError("Some baseline test IDs did not match the source dataset.")
if len(experiment_df) < MAIN_SAMPLE_SIZE:
    raise ValueError(f"Need {MAIN_SAMPLE_SIZE} test rows, but found {len(experiment_df)}.")

main_sample = experiment_df.sample(n=MAIN_SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
smoke_sample = main_sample.head(DRY_RUN_SIZE).copy()
profile_fields = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod", "MonthlyCharges", "TotalCharges",
]
missing_fields = sorted(set(profile_fields) - set(experiment_df.columns))
if missing_fields:
    raise ValueError(f"Source dataset is missing required profile fields: {missing_fields}")
print(f"Loaded {len(experiment_df)} test rows; selected {len(main_sample)} rows, with {len(smoke_sample)} for the smoke test.")
display(main_sample[["customerID", "actual_churn", "baseline_predicted"]].head())

Loaded 1409 test rows; selected 200 rows, with 8 for the smoke test.


,customerID,actual_churn,baseline_predicted
0,7054-ENNGU,No,No
1,1310-QRITU,No,Yes
2,8347-GDTMP,No,No
3,4283-FUTGF,Yes,Yes
4,1525-LNLOJ,No,No


## Build the local experiment sample

The prepared baseline test predictions are joined to source features by customer ID. A deterministic 200-row sample and its first eight records are selected; this keeps the LLM evaluation on held-out customers and enables direct comparison against the baseline and actual outcomes.

In [13]:
def build_profile_prompt(row: pd.Series) -> str:
    """Build a feature-only prompt; identifiers, labels, and baseline outputs are excluded."""
    profile = {
        name: ("Unknown" if pd.isna(row[name]) else row[name])
        for name in profile_fields
    }
    profile_text = "\n".join(f"{name}: {value}" for name, value in profile.items())
    return (
        "Estimate whether this customer will churn based only on the profile below. "
        "Return exactly one JSON object with keys predicted_label and rationale. "
        "predicted_label must be exactly Yes or No. rationale must be one concise sentence. "
        "Do not include markdown fences or any text outside the JSON object.\n\n"
        f"Customer profile:\n{profile_text}"
    )


def parse_classification(raw_text: str) -> tuple[str | None, str | None, str | None]:
    """Return normalized label, rationale, and parse error, if any."""
    try:
        cleaned = raw_text.strip()
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned, flags=re.IGNORECASE)
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match is None:
            raise ValueError("No JSON object found")
        payload = json.loads(match.group(0))
        label = str(payload.get("predicted_label", "")).strip().casefold()
        rationale = str(payload.get("rationale", "")).strip()
        if label not in {"yes", "no"}:
            raise ValueError("predicted_label must be Yes or No")
        if not rationale:
            raise ValueError("rationale is empty")
        return label.title(), rationale, None
    except (ValueError, json.JSONDecodeError, AttributeError, TypeError) as exc:
        return None, None, f"{type(exc).__name__}: {exc}"


def classify_customer(row: pd.Series, model_name: str = MODEL_NAME) -> dict:
    """Call Anthropic with retry/backoff and return an auditable record."""
    started_at = datetime.now(timezone.utc).isoformat()
    start = time.perf_counter()
    raw_text = ""
    input_tokens = None
    output_tokens = None
    error = None
    retryable_errors = (
        anthropic.APIConnectionError,
        anthropic.APITimeoutError,
        anthropic.RateLimitError,
        anthropic.InternalServerError,
    )
    for attempt in range(MAX_RETRIES):
        try:
            response = client.messages.create(
                model=model_name,
                max_tokens=120,
                temperature=0,
                system="You are a careful churn classification assistant. Follow the requested JSON schema exactly.",
                messages=[{"role": "user", "content": build_profile_prompt(row)}],
            )
            raw_text = "".join(block.text for block in response.content if getattr(block, "type", None) == "text")
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens
            break
        except retryable_errors as exc:
            error = f"{type(exc).__name__}: transient API failure"
            if attempt + 1 < MAX_RETRIES:
                time.sleep(INITIAL_BACKOFF_SECONDS * (2**attempt))
        except anthropic.APIStatusError as exc:
            detail = str(getattr(exc, "message", str(exc))).replace(api_key, "[REDACTED]")
            error = f"{type(exc).__name__}: API request rejected (status {exc.status_code}): {detail[:500]}"
            break
        except Exception as exc:
            # Do not persist arbitrary exception text, which could contain request details.
            error = f"{type(exc).__name__}: unexpected request failure"
            break

    latency_seconds = time.perf_counter() - start
    label, rationale, parse_error = parse_classification(raw_text) if raw_text else (None, None, None)
    if parse_error:
        error = parse_error
    return {
        "customerID": row["customerID"],
        "actual_churn": row["actual_churn"],
        "baseline_predicted": row["baseline_predicted"],
        "predicted_label": label,
        "rationale": rationale,
        "raw_response": raw_text,
        "model": model_name,
        "started_at_utc": started_at,
        "latency_seconds": latency_seconds,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "status": "ok" if label in {"Yes", "No"} and rationale else "error",
        "error": error,
    }

## Prompt, parse, and record the small pilot

The helper builds a compact feature-only profile, requests a constrained JSON answer, validates the returned label/rationale, and captures latency, token counts, raw response, and safe error details. The next cell performs the first small-format pilot and records failures for review.

In [10]:
smoke_records = []
for position, (_, sample_row) in enumerate(smoke_sample.iterrows(), start=1):
    result = classify_customer(sample_row)
    smoke_records.append(result)
    print(f"Smoke row {position}/{len(smoke_sample)}: {result['status']}; latency={result['latency_seconds']:.2f}s")
    print("Raw response:", result["raw_response"] or "<empty>")

smoke_results = pd.DataFrame(smoke_records)
smoke_output_path = project_root / "data" / "processed" / "haiku_pilot_predictions.csv"
smoke_output_path.parent.mkdir(parents=True, exist_ok=True)
smoke_results.to_csv(smoke_output_path, index=False)
smoke_valid = (
    len(smoke_results) == DRY_RUN_SIZE
    and smoke_results["status"].eq("ok").all()
    and smoke_results["predicted_label"].isin(["Yes", "No"]).all()
    and smoke_results["rationale"].fillna("").str.strip().ne("").all()
)
print(f"Smoke validation passed: {smoke_valid}. Pilot results saved to {smoke_output_path}.")
display(smoke_results[["customerID", "predicted_label", "rationale", "latency_seconds", "input_tokens", "output_tokens", "status", "error"]])

Smoke row 1/8: error; latency=0.31s
Raw response: <empty>
Smoke row 2/8: error; latency=0.23s
Raw response: <empty>
Smoke row 3/8: error; latency=0.23s
Raw response: <empty>
Smoke row 4/8: error; latency=0.21s
Raw response: <empty>
Smoke row 5/8: error; latency=0.23s
Raw response: <empty>
Smoke row 6/8: error; latency=0.20s
Raw response: <empty>
Smoke row 7/8: error; latency=0.25s
Raw response: <empty>
Smoke row 8/8: error; latency=0.21s
Raw response: <empty>
Smoke validation passed: False. Pilot results saved to c:\Users\cdasuncion\churn-llm-comparison\data\processed\haiku_pilot_predictions.csv.


,customerID,predicted_label,rationale,latency_seconds,input_tokens,output_tokens,status,error
0,7054-ENNGU,None,None,0.307104,None,None,error,BadRequestError: API request rejected (status ...
1,1310-QRITU,None,None,0.234497,None,None,error,BadRequestError: API request rejected (status ...
2,8347-GDTMP,None,None,0.230926,None,None,error,BadRequestError: API request rejected (status ...
3,4283-FUTGF,None,None,0.213242,None,None,error,BadRequestError: API request rejected (status ...
4,1525-LNLOJ,None,None,0.229919,None,None,error,BadRequestError: API request rejected (status ...
5,1574-DYCWE,None,None,0.202700,None,None,error,BadRequestError: API request rejected (status ...
6,7663-CUXZB,None,None,0.245265,None,None,error,BadRequestError: API request rejected (status ...
7,7410-KTVFV,None,None,0.206534,None,None,error,BadRequestError: API request rejected (status ...


## Gate scaling on pilot quality

The pilot must return valid `Yes`/`No` labels and non-empty rationales for every row. If it fails, the workflow turns off the larger batch so malformed outputs or billing/API issues are handled before additional requests are made.

In [14]:
if not smoke_valid:
    RUN_FULL_BATCH = False
    print("STOP: the smoke test failed validation. Inspect the saved errors and resolve them before enabling the full batch.")
else:
    print("Smoke test is valid. Review raw responses and cost/latency before opting in to 200-row inference.")

STOP: the smoke test failed validation. Inspect the saved errors and resolve them before enabling the full batch.


## Main 200-row Haiku batch (explicit opt-in)

Leave `RUN_FULL_BATCH = False` while inspecting the pilot. After it passes your review, change the toggle above to `True` and rerun this cell. It will reuse the 8 pilot responses and make only the remaining API calls; expect additional billable usage.

In [16]:
batch_results = None
if RUN_FULL_BATCH and not smoke_valid:
    print("Full batch blocked because the smoke test did not pass validation.")
elif RUN_FULL_BATCH:
    prior_records = smoke_results.to_dict(orient="records")
    completed_ids = {record["customerID"] for record in prior_records}
    pending = main_sample.loc[~main_sample["customerID"].isin(completed_ids)]
    all_records = list(prior_records)
    for position, (_, batch_row) in enumerate(pending.iterrows(), start=len(prior_records) + 1):
        result = classify_customer(batch_row, model_name=MODEL_NAME)
        all_records.append(result)
        if position % 10 == 0 or position == MAIN_SAMPLE_SIZE:
            print(f"Processed {position}/{MAIN_SAMPLE_SIZE}; latest status={result['status']}")
        if REQUEST_PAUSE_SECONDS > 0:
            time.sleep(REQUEST_PAUSE_SECONDS)

    batch_results = pd.DataFrame(all_records)
    batch_output_path = project_root / "data" / "processed" / "haiku_200_predictions.csv"
    batch_results.to_csv(batch_output_path, index=False)
    batch_metadata = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": MODEL_NAME,
        "sample_size_requested": MAIN_SAMPLE_SIZE,
        "sample_size_processed": len(batch_results),
        "smoke_rows_reused": len(prior_records),
        "random_seed": RANDOM_SEED,
        "retry_limit": MAX_RETRIES,
        "error_rows": int(batch_results["status"].ne("ok").sum()),
        "output_csv": str(batch_output_path.relative_to(project_root)),
    }
    metadata_path = project_root / "data" / "processed" / "haiku_200_metadata.json"
    metadata_path.write_text(json.dumps(batch_metadata, indent=2), encoding="utf-8")
    print(f"Saved row results to {batch_output_path} and metadata to {metadata_path}.")
else:
    print("Full batch skipped. It remains blocked until a smoke test passes; after resolving API access, rerun the pilot, review it, then enable RUN_FULL_BATCH.")

Full batch skipped. It remains blocked until a smoke test passes; after resolving API access, rerun the pilot, review it, then enable RUN_FULL_BATCH.


## Evaluate predictions on comparable cases

Metrics use rows with valid LLM labels, with baseline results calculated on the same subset. The confusion matrices and disagreement list show where the models differ; when the opt-in 200-row batch runs, row-level results and aggregate metrics are saved locally.

In [17]:
evaluation_results = batch_results if batch_results is not None else smoke_results
valid_rows = evaluation_results.loc[
    evaluation_results["status"].eq("ok")
    & evaluation_results["predicted_label"].isin(["Yes", "No"])
].copy()

print(f"Evaluation sample: {len(evaluation_results)} rows; valid LLM responses: {len(valid_rows)}; failures: {len(evaluation_results) - len(valid_rows)}")
if len(valid_rows):
    def metric_row(name: str, predictions: pd.Series) -> dict:
        return {
            "model": name,
            "accuracy": accuracy_score(valid_rows["actual_churn"], predictions),
            "precision_yes": precision_score(valid_rows["actual_churn"], predictions, pos_label="Yes", zero_division=0),
            "recall_yes": recall_score(valid_rows["actual_churn"], predictions, pos_label="Yes", zero_division=0),
            "f1_yes": f1_score(valid_rows["actual_churn"], predictions, pos_label="Yes", zero_division=0),
        }

    comparison_metrics = pd.DataFrame(
        [
            metric_row("Claude Haiku", valid_rows["predicted_label"]),
            metric_row("Logistic regression baseline", valid_rows["baseline_predicted"]),
        ]
    )
    agreement_rate = (valid_rows["predicted_label"] == valid_rows["baseline_predicted"]).mean()
    print(f"Haiku/baseline agreement on valid rows: {agreement_rate:.3f}")
    display(comparison_metrics.round(4))
    print("Haiku confusion matrix (actual rows, predicted columns; No, Yes):")
    print(confusion_matrix(valid_rows["actual_churn"], valid_rows["predicted_label"], labels=["No", "Yes"]))
    print("Baseline confusion matrix on the same valid rows (actual rows, predicted columns; No, Yes):")
    print(confusion_matrix(valid_rows["actual_churn"], valid_rows["baseline_predicted"], labels=["No", "Yes"]))
    disagreements = valid_rows.loc[
        valid_rows["predicted_label"] != valid_rows["baseline_predicted"],
        ["customerID", "actual_churn", "baseline_predicted", "predicted_label", "rationale"],
    ]
    print(f"Disagreements for review: {len(disagreements)}")
    display(disagreements.head(20))
else:
    comparison_metrics = pd.DataFrame()
    agreement_rate = float("nan")
    print("No valid LLM responses available for metric calculation.")

if batch_results is not None:
    comparison_path = project_root / "data" / "processed" / "haiku_200_comparison.csv"
    batch_results.to_csv(comparison_path, index=False)
    metrics_summary_path = project_root / "data" / "processed" / "haiku_200_metrics.json"
    metrics_summary_path.write_text(
        json.dumps(
            {
                "sample_size_total": len(evaluation_results),
                "sample_size_valid_llm": len(valid_rows),
                "llm_failures": int(evaluation_results["status"].ne("ok").sum()),
                "baseline_agreement": agreement_rate,
                "metrics": comparison_metrics.to_dict(orient="records"),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Saved comparison data to {comparison_path} and metrics to {metrics_summary_path}.")

Evaluation sample: 8 rows; valid LLM responses: 0; failures: 8
No valid LLM responses available for metric calculation.


## Optional Sonnet side-sample (25 rows)

This is a separate, opt-in finding, not part of the 200-row Haiku run. It uses the same locally loaded API key and classifier pipeline, changing only the model and selecting 25 rows.

In [18]:
sonnet_results = None
if RUN_SONNET_SIDE_EXPERIMENT:
    sonnet_sample = main_sample.sample(n=SONNET_SAMPLE_SIZE, random_state=RANDOM_SEED + 1)
    sonnet_records = []
    for position, (_, sonnet_row) in enumerate(sonnet_sample.iterrows(), start=1):
        result = classify_customer(sonnet_row, model_name=SONNET_MODEL_NAME)
        sonnet_records.append(result)
        if position % 5 == 0 or position == SONNET_SAMPLE_SIZE:
            print(f"Sonnet side-sample processed {position}/{SONNET_SAMPLE_SIZE}; latest status={result['status']}")
        if REQUEST_PAUSE_SECONDS > 0:
            time.sleep(REQUEST_PAUSE_SECONDS)

    sonnet_results = pd.DataFrame(sonnet_records)
    sonnet_path = project_root / "data" / "processed" / "sonnet_25_predictions.csv"
    sonnet_results.to_csv(sonnet_path, index=False)
    sonnet_meta = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": SONNET_MODEL_NAME,
        "sample_size_requested": SONNET_SAMPLE_SIZE,
        "sample_size_processed": len(sonnet_results),
        "random_seed": RANDOM_SEED + 1,
        "error_rows": int(sonnet_results["status"].ne("ok").sum()),
        "output_csv": str(sonnet_path.relative_to(project_root)),
    }
    (project_root / "data" / "processed" / "sonnet_25_metadata.json").write_text(
        json.dumps(sonnet_meta, indent=2), encoding="utf-8"
    )
    display(sonnet_results[["customerID", "actual_churn", "predicted_label", "rationale", "status", "error"]])
else:
    print("Sonnet side experiment skipped. Set RUN_SONNET_SIDE_EXPERIMENT = True to run the 25-row sample.")

Sonnet side experiment skipped. Set RUN_SONNET_SIDE_EXPERIMENT = True to run the 25-row sample.


## Compare optional Sonnet results

The side experiment remains off by default and is limited to 25 rows. If run, this cell reports Sonnet's metrics against the actual labels alongside baseline performance on those same successfully classified records.

In [20]:
if sonnet_results is not None:
    sonnet_valid = sonnet_results.loc[
        sonnet_results["status"].eq("ok")
        & sonnet_results["predicted_label"].isin(["Yes", "No"])
    ].copy()
    if len(sonnet_valid):
        sonnet_metrics = pd.DataFrame(
            [
                {
                    "model": "Claude Sonnet",
                    "accuracy": accuracy_score(sonnet_valid["actual_churn"], sonnet_valid["predicted_label"]),
                    "precision_yes": precision_score(sonnet_valid["actual_churn"], sonnet_valid["predicted_label"], pos_label="Yes", zero_division=0),
                    "recall_yes": recall_score(sonnet_valid["actual_churn"], sonnet_valid["predicted_label"], pos_label="Yes", zero_division=0),
                    "f1_yes": f1_score(sonnet_valid["actual_churn"], sonnet_valid["predicted_label"], pos_label="Yes", zero_division=0),
                    "baseline_agreement": (sonnet_valid["predicted_label"] == sonnet_valid["baseline_predicted"]).mean(),
                },
                {
                    "model": "Logistic regression baseline",
                    "accuracy": accuracy_score(sonnet_valid["actual_churn"], sonnet_valid["baseline_predicted"]),
                    "precision_yes": precision_score(sonnet_valid["actual_churn"], sonnet_valid["baseline_predicted"], pos_label="Yes", zero_division=0),
                    "recall_yes": recall_score(sonnet_valid["actual_churn"], sonnet_valid["baseline_predicted"], pos_label="Yes", zero_division=0),
                    "f1_yes": f1_score(sonnet_valid["actual_churn"], sonnet_valid["baseline_predicted"], pos_label="Yes", zero_division=0),
                    "baseline_agreement": 1.0,
                },
            ]
        )
        display(sonnet_metrics.round(4))
        print("Sonnet confusion matrix (actual rows, predicted columns; No, Yes):")
        print(confusion_matrix(sonnet_valid["actual_churn"], sonnet_valid["predicted_label"], labels=["No", "Yes"]))
        sonnet_metrics_path = project_root / "data" / "processed" / "sonnet_25_metrics.json"
        sonnet_metrics_path.write_text(
            json.dumps(
                {
                    "valid_rows": len(sonnet_valid),
                    "failures": int(sonnet_results["status"].ne("ok").sum()),
                    "metrics": sonnet_metrics.to_dict(orient="records"),
                },
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"Saved Sonnet side-sample metrics to {sonnet_metrics_path}.")
    else:
        print("No valid Sonnet outputs were available for metric calculation.")

In [19]:
readme_rationale = """## Why Haiku and Claude?

Haiku is a cost- and latency-conscious fit for this focused task: predicting Yes/No from a short customer profile with one concise rationale does not require heavyweight multi-step reasoning. Claude was the practical provider choice given existing access. Comparing other providers is a natural next step.

The main LLM comparison is scoped to a reproducible 200-row sample after an 8-row smoke test. A future Sonnet check should remain a separate 20–30-row side-sample, using the same local API key and changing only the model setting.
"""
print("README-ready rationale:\n")
print(readme_rationale)

README-ready rationale:

## Why Haiku and Claude?

Haiku is a cost- and latency-conscious fit for this focused task: predicting Yes/No from a short customer profile with one concise rationale does not require heavyweight multi-step reasoning. Claude was the practical provider choice given existing access. Comparing other providers is a natural next step.

The main LLM comparison is scoped to a reproducible 200-row sample after an 8-row smoke test. A future Sonnet check should remain a separate 20–30-row side-sample, using the same local API key and changing only the model setting.



## Document the model choice

This reproducible note summarizes why a fast, cost-conscious Haiku model fits a short binary prediction task, why Claude is the practical provider for this project, and why a different provider is a natural future comparison.

## Natural-language summaries and requested response format

This section builds three local test summaries from the reproducible test sample. The live API check is controlled separately so no more requests are sent until API billing access is available.

In [21]:
def _service_phrase(value: object, service_name: str) -> str:
    """Describe a Yes/No service value in conversational form."""
    normalized = "" if pd.isna(value) else str(value).strip().casefold()
    if normalized == "yes":
        return f"has {service_name}"
    if normalized == "no":
        return f"has no {service_name}"
    if normalized == "no internet service":
        return f"does not have {service_name} because they have no internet service"
    return f"has an unknown {service_name} status"


def row_to_summary(row: pd.Series) -> str:
    """Convert a Telco churn record into a short natural-language summary."""
    contract_value = str(row.get("Contract", "unknown contract")).strip().casefold()
    contract = contract_value if contract_value else "unknown contract"

    tenure_value = row.get("tenure")
    tenure = "an unknown number of months" if pd.isna(tenure_value) else f"{int(tenure_value)} months"
    charge_value = row.get("MonthlyCharges")
    monthly_charge = "an unknown amount" if pd.isna(charge_value) else f"${float(charge_value):.2f}"

    security = str(row.get("OnlineSecurity", "")).strip()
    support = str(row.get("TechSupport", "")).strip()
    if security == "No internet service" and support == "No internet service":
        support_phrase = "does not have internet service, so online security and tech support are unavailable"
    elif security == "No" and support == "No":
        support_phrase = "has no online security or tech support"
    else:
        security_phrase = _service_phrase(row.get("OnlineSecurity"), "online security")
        support_service_phrase = _service_phrase(row.get("TechSupport"), "tech support")
        support_phrase = f"{security_phrase} and {support_service_phrase}"

    payment_value = row.get("PaymentMethod")
    payment_method = "an unknown payment method" if pd.isna(payment_value) else str(payment_value).strip().casefold()
    return (
        f"This customer has a {contract} contract, has been with the company for {tenure}, "
        f"pays {monthly_charge} per month, {support_phrase}, and pays by {payment_method}."
    )


example_rows = main_sample.head(3).copy()
example_rows["summary"] = example_rows.apply(row_to_summary, axis=1)
for example_number, (_, example_row) in enumerate(example_rows.iterrows(), start=1):
    print(f"Example {example_number}: {example_row['summary']}")

Example 1: This customer has a month-to-month contract, has been with the company for 9 months, pays $50.85 per month, has no online security or tech support, and pays by mailed check.
Example 2: This customer has a month-to-month contract, has been with the company for 18 months, pays $50.30 per month, has no online security or tech support, and pays by electronic check.
Example 3: This customer has a two year contract, has been with the company for 64 months, pays $19.45 per month, does not have internet service, so online security and tech support are unavailable, and pays by credit card (automatic).


## Send only the natural-language summary

`predict_with_llm()` uses Claude Haiku with a churn-prediction system prompt and asks for the exact `PREDICTION: ... / REASON: ...` format. The parser checks the label and reason separately; local synthetic responses exercise parsing without API calls before the live examples are tested.

In [22]:
PREDICTION_MODEL = "claude-haiku-4-5-20251001"


def parse_llm_response(response_text: str) -> dict[str, str | None]:
    """Parse the required PREDICTION / REASON response format."""
    pattern = r"^\s*PREDICTION:\s*(Yes|No)\s*/\s*REASON:\s*(.+?)\s*$"
    match = re.match(pattern, response_text, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        return {"predicted_churn": None, "reason": None, "parse_error": "Response did not match required format."}
    reason = " ".join(match.group(2).split())
    if not reason:
        return {"predicted_churn": None, "reason": None, "parse_error": "Reason was empty."}
    return {"predicted_churn": match.group(1).title(), "reason": reason, "parse_error": None}


def predict_with_llm(summary_text: str) -> dict[str, str | None]:
    """Ask Claude Haiku to classify a single summary and parse its exact-format answer."""
    try:
        response = client.messages.create(
            model=PREDICTION_MODEL,
            max_tokens=100,
            temperature=0,
            system=(
                "You are performing customer churn prediction. Based only on the supplied customer summary, "
                "estimate whether the customer will leave the service. Do not claim certainty."
            ),
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Customer summary:\n{summary_text}\n\n"
                        "Will this customer churn? Answer in this exact format: "
                        "PREDICTION: Yes or No / REASON: one short sentence."
                    ),
                }
            ],
        )
        raw_response = "".join(
            block.text for block in response.content if getattr(block, "type", None) == "text"
        ).strip()
        parsed = parse_llm_response(raw_response)
        return {
            "predicted_churn": parsed["predicted_churn"],
            "reason": parsed["reason"],
            "raw_response": raw_response,
            "error": parsed["parse_error"],
        }
    except anthropic.APIStatusError as exc:
        detail = str(getattr(exc, "message", str(exc))).replace(api_key, "[REDACTED]")
        return {
            "predicted_churn": None,
            "reason": None,
            "raw_response": "",
            "error": f"{type(exc).__name__} status {exc.status_code}: {detail[:400]}",
        }
    except Exception as exc:
        return {
            "predicted_churn": None,
            "reason": None,
            "raw_response": "",
            "error": f"{type(exc).__name__}: request failed",
        }


# Offline parser checks exercise both supported labels without making API requests.
parser_checks = [
    parse_llm_response("PREDICTION: Yes / REASON: The short tenure and monthly contract suggest elevated churn risk."),
    parse_llm_response("PREDICTION: No / REASON: The long-term contract suggests lower churn risk."),
]
assert all(check["predicted_churn"] in {"Yes", "No"} and not check["parse_error"] for check in parser_checks)
print("Response parser checks passed for both Yes and No formats.")

Response parser checks passed for both Yes and No formats.


## Validate on three held-out customers

The three sample profiles are sent individually to the API, and the notebook prints each parsed label, rationale, or safe API/format error. Any failed request keeps the 200-row batch locked until you obtain valid outputs and review them.

In [23]:
THREE_ROW_LIVE_TEST = True
three_row_results = []
if THREE_ROW_LIVE_TEST:
    for example_number, (_, example_row) in enumerate(example_rows.iterrows(), start=1):
        llm_response = predict_with_llm(example_row["summary"])
        three_row_results.append(
            {
                "customerID": example_row["customerID"],
                "summary": example_row["summary"],
                "predicted_churn": llm_response["predicted_churn"],
                "reason": llm_response["reason"],
                "error": llm_response["error"],
            }
        )
        print(f"Live test {example_number}/3:", llm_response["predicted_churn"] or "no valid prediction")
        if llm_response["reason"]:
            print("Reason:", llm_response["reason"])
        if llm_response["error"]:
            print("Error:", llm_response["error"])
        time.sleep(0.25)

three_row_results_df = pd.DataFrame(three_row_results)
THREE_ROW_TEST_PASSED = (
    len(three_row_results_df) == 3
    and three_row_results_df["predicted_churn"].isin(["Yes", "No"]).all()
    and three_row_results_df["reason"].fillna("").str.strip().ne("").all()
)
print(f"Three-row live test passed: {THREE_ROW_TEST_PASSED}")
display(three_row_results_df)

Live test 1/3: no valid prediction
Error: BadRequestError status 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Cfb5mSSQ7KB37JGJvFMaG'}
Live test 2/3: no valid prediction
Error: BadRequestError status 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Cfb5mUuyAHXdNsvWzg1QL'}
Live test 3/3: no valid prediction
Error: BadRequestError status 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Cfb5mWw1if2XzfVGPfpoj'}
Three-r

,customerID,summary,predicted_churn,reason,error
0,7054-ENNGU,"This customer has a month-to-month contract, h...",None,None,BadRequestError status 400: Error code: 400 - ...
1,1310-QRITU,"This customer has a month-to-month contract, h...",None,None,BadRequestError status 400: Error code: 400 - ...
2,8347-GDTMP,"This customer has a two year contract, has bee...",None,None,BadRequestError status 400: Error code: 400 - ...


## Optional 200-row test-set inference

The batch is gated on a successful 3-row live test and remains disabled by default. After checking the three outputs and ensuring billing is available, set `RUN_200_ROW_BATCH = True` and run the batch cell. It samples the test set with `random_state=42`, sleeps briefly between requests, and reports progress every 20 rows.

In [24]:
RUN_200_ROW_BATCH = False
llm_batch_predictions = None
if not THREE_ROW_TEST_PASSED:
    print("200-row run blocked: first obtain three valid live predictions from the test rows.")
elif not RUN_200_ROW_BATCH:
    print("200-row run not started. Review the live test outputs, then set RUN_200_ROW_BATCH = True to proceed.")
else:
    llm_test_sample = experiment_df.sample(n=200, random_state=42).copy()
    batch_rows = []
    for row_number, (_, test_row) in enumerate(llm_test_sample.iterrows(), start=1):
        summary_text = row_to_summary(test_row)
        llm_result = predict_with_llm(summary_text)
        batch_rows.append(
            {
                "customerID": test_row["customerID"],
                "actual_churn": test_row["actual_churn"],
                "llm_predicted": llm_result["predicted_churn"],
                "llm_reason": llm_result["reason"] or llm_result["error"],
            }
        )
        if row_number % 20 == 0 or row_number == len(llm_test_sample):
            print(f"Completed {row_number}/{len(llm_test_sample)} rows")
        time.sleep(0.25)

    llm_batch_predictions = pd.DataFrame(
        batch_rows,
        columns=["customerID", "actual_churn", "llm_predicted", "llm_reason"],
    )
    llm_output_path = project_root / "data" / "processed" / "llm_predictions.csv"
    llm_output_path.parent.mkdir(parents=True, exist_ok=True)
    llm_batch_predictions.to_csv(llm_output_path, index=False)
    print(f"Saved {len(llm_batch_predictions)} predictions to {llm_output_path}")
    display(llm_batch_predictions.head())

200-row run blocked: first obtain three valid live predictions from the test rows.


## Score valid LLM outputs

This cell calculates accuracy, precision, recall, F1, and the confusion matrix only for records with a valid `Yes`/`No` prediction. Failed rows remain in the CSV for auditability but are excluded from classification metrics.

In [25]:
if llm_batch_predictions is None:
    print("LLM metrics are pending a successful 200-row inference run.")
else:
    valid_predictions = llm_batch_predictions.loc[
        llm_batch_predictions["llm_predicted"].isin(["Yes", "No"])
    ].copy()
    print(f"Valid LLM predictions: {len(valid_predictions)}/{len(llm_batch_predictions)}")
    if valid_predictions.empty:
        print("No valid predictions available; metrics cannot be calculated.")
    else:
        y_true = valid_predictions["actual_churn"]
        y_predicted = valid_predictions["llm_predicted"]
        print(f"Accuracy:  {accuracy_score(y_true, y_predicted):.4f}")
        print(f"Precision: {precision_score(y_true, y_predicted, pos_label='Yes', zero_division=0):.4f}")
        print(f"Recall:    {recall_score(y_true, y_predicted, pos_label='Yes', zero_division=0):.4f}")
        print(f"F1:        {f1_score(y_true, y_predicted, pos_label='Yes', zero_division=0):.4f}")
        print("Confusion matrix (rows=actual, columns=predicted; labels No, Yes):")
        print(confusion_matrix(y_true, y_predicted, labels=["No", "Yes"]))

LLM metrics are pending a successful 200-row inference run.


## Compare baseline and LLM predictions

The final analysis joins the baseline and LLM CSVs on `customerID`, verifies any duplicated ground-truth labels, and uses the source dataset to reproduce Phase 2's tenure bins and full-dataset MonthlyCharges quartiles. It reports complete-text disagreement examples, segment disagreement rates, which model was correct on disagreements, and an optional score that weights false negatives five times false positives. Invalid LLM labels are counted and excluded from agreement/metrics rather than being treated as genuine model disagreements.

In [26]:
import numpy as np

processed_dir = project_root / "data" / "processed"
baseline_csv = processed_dir / "baseline_predictions.csv"
llm_csv = processed_dir / "llm_predictions.csv"
source_csv = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
missing_inputs = [path.name for path in (baseline_csv, llm_csv, source_csv) if not path.exists()]
analysis_ready = False

if missing_inputs:
    print(f"Cannot build the comparison yet; missing input file(s): {', '.join(missing_inputs)}")
    print("Generate the LLM predictions first, then rerun this analysis cell.")
else:
    baseline_input = pd.read_csv(baseline_csv, dtype={"customerID": "string"})
    llm_input = pd.read_csv(llm_csv, dtype={"customerID": "string"})
    source_features = pd.read_csv(
        source_csv,
        usecols=["customerID", "Contract", "tenure", "MonthlyCharges"],
        dtype={"customerID": "string"},
    )
    required_baseline = {"customerID", "actual_churn", "baseline_predicted"}
    required_llm = {"customerID", "llm_predicted", "llm_reason"}
    if not required_baseline.issubset(baseline_input.columns):
        raise ValueError(f"Baseline predictions must contain columns: {sorted(required_baseline)}")
    if not required_llm.issubset(llm_input.columns):
        raise ValueError(f"LLM predictions must contain columns: {sorted(required_llm)}")

    baseline_columns = baseline_input[["customerID", "actual_churn", "baseline_predicted"]].copy()
    llm_columns = llm_input[["customerID", "llm_predicted", "llm_reason"]].copy()
    if "actual_churn" in llm_input.columns:
        llm_columns["llm_actual_churn"] = llm_input["actual_churn"]

    merged_all = baseline_columns.merge(
        llm_columns,
        on="customerID",
        how="inner",
        validate="one_to_one",
    )
    if "llm_actual_churn" in merged_all.columns:
        actual_disagreement = (
            merged_all["actual_churn"].astype("string").str.strip()
            != merged_all["llm_actual_churn"].astype("string").str.strip()
        )
        if actual_disagreement.any():
            raise ValueError(f"Ground-truth churn labels differ for {int(actual_disagreement.sum())} joined customers.")

    for label_column in ("actual_churn", "baseline_predicted", "llm_predicted"):
        merged_all[label_column] = merged_all[label_column].astype("string").str.strip().str.title()
    valid_mask = (
        merged_all["actual_churn"].isin(["Yes", "No"])
        & merged_all["baseline_predicted"].isin(["Yes", "No"])
        & merged_all["llm_predicted"].isin(["Yes", "No"])
    )
    invalid_prediction_count = int((~valid_mask).sum())
    merged_valid = merged_all.loc[valid_mask].copy()
    merged_valid["agree"] = merged_valid["baseline_predicted"].eq(merged_valid["llm_predicted"])

    # Match Phase 2: tenure intervals and quartiles ranked across the full raw dataset.
    source_features["tenure_bucket"] = pd.cut(
        pd.to_numeric(source_features["tenure"], errors="coerce"),
        bins=[-1, 12, 24, 48, np.inf],
        labels=["0-12 months", "13-24 months", "25-48 months", "49+ months"],
    )
    source_features["cost_quartile"] = pd.qcut(
        pd.to_numeric(source_features["MonthlyCharges"], errors="coerce").rank(method="first"),
        q=4,
        labels=["Q1", "Q2", "Q3", "Q4"],
    )
    comparison = merged_valid.merge(
        source_features[["customerID", "Contract", "tenure_bucket", "cost_quartile"]],
        on="customerID",
        how="left",
        validate="one_to_one",
    )
    if comparison[["Contract", "tenure_bucket", "cost_quartile"]].isna().any().any():
        raise ValueError("Some joined prediction rows are missing segment fields from the source dataset.")

    comparison["agree"] = comparison["baseline_predicted"].eq(comparison["llm_predicted"])
    comparison_table = comparison[
        ["customerID", "actual_churn", "baseline_predicted", "llm_predicted", "llm_reason", "agree"]
    ].copy()
    comparison_output = processed_dir / "model_comparison.csv"
    comparison_table.to_csv(comparison_output, index=False)
    analysis_ready = True

    disagreements = comparison.loc[~comparison["agree"]].copy()
    print(f"Joined valid predictions: {len(comparison)}; excluded rows with invalid/missing labels: {invalid_prediction_count}")
    print(f"Baseline/LLM disagreements: {len(disagreements)}")
    print(f"Saved comparison table to {comparison_output}")
    print("First 15 disagreement examples (complete text, including llm_reason):")
    with pd.option_context("display.max_colwidth", None, "display.width", 0):
        display(disagreements.head(15).to_string(index=False))

Cannot build the comparison yet; missing input file(s): llm_predictions.csv
Generate the LLM predictions first, then rerun this analysis cell.


## Summarize segment outcomes and asymmetric costs

Each segment's disagreement rate uses valid model predictions in that segment as its denominator. On disagreeing rows, exactly one model should match the actual binary label; the table shows counts, percentages, and the model correct more often. The optional cost score separately evaluates all valid joined rows, charging 5 for a missed churner and 1 for a false alarm.

In [27]:
if analysis_ready:
    segment_levels = {
        "Contract": ["Month-to-month", "One year", "Two year"],
        "tenure_bucket": ["0-12 months", "13-24 months", "25-48 months", "49+ months"],
        "cost_quartile": ["Q1", "Q2", "Q3", "Q4"],
    }
    segment_rows = []
    for segment_name, levels in segment_levels.items():
        for level in levels:
            segment_data = comparison.loc[comparison[segment_name].astype("string").eq(level)]
            disagreement_data = segment_data.loc[~segment_data["agree"]]
            baseline_correct = int(
                disagreement_data["baseline_predicted"].eq(disagreement_data["actual_churn"]).sum()
            )
            llm_correct = int(
                disagreement_data["llm_predicted"].eq(disagreement_data["actual_churn"]).sum()
            )
            disagreement_count = len(disagreement_data)
            segment_rows.append(
                {
                    "segment_dimension": segment_name,
                    "segment": level,
                    "rows_with_valid_predictions": len(segment_data),
                    "disagreement_count": disagreement_count,
                    "disagreement_rate": (
                        disagreement_count / len(segment_data) if len(segment_data) else np.nan
                    ),
                    "baseline_correct_on_disagreements": baseline_correct,
                    "llm_correct_on_disagreements": llm_correct,
                    "baseline_correct_pct_of_disagreements": (
                        baseline_correct / disagreement_count if disagreement_count else np.nan
                    ),
                    "llm_correct_pct_of_disagreements": (
                        llm_correct / disagreement_count if disagreement_count else np.nan
                    ),
                    "correct_more_often_on_disagreements": (
                        "Baseline" if baseline_correct > llm_correct
                        else "LLM" if llm_correct > baseline_correct
                        else "Tie" if disagreement_count else "No disagreements"
                    ),
                }
            )

    segment_analysis = pd.DataFrame(segment_rows)
    print("Disagreement rate and relative correctness by contract, tenure bucket, and monthly-charge quartile:")
    display(
        segment_analysis.style.format(
            {
                "disagreement_rate": "{:.1%}",
                "baseline_correct_pct_of_disagreements": "{:.1%}",
                "llm_correct_pct_of_disagreements": "{:.1%}",
            },
            na_rep="—",
        )
    )
    segment_output = processed_dir / "disagreement_by_segment.csv"
    segment_analysis.to_csv(segment_output, index=False)
    print(f"Saved segment analysis to {segment_output}")

    # Optional cost-sensitive score: 1 - observed weighted penalty / worst-case penalty.
    # False negatives cost 5; false positives cost 1; correct predictions cost 0.
    def cost_weighted_summary(actual: pd.Series, predicted: pd.Series, model_name: str) -> dict:
        tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=["No", "Yes"]).ravel()
        weighted_penalty = int(5 * fn + fp)
        worst_case_penalty = int(5 * actual.eq("Yes").sum() + actual.eq("No").sum())
        return {
            "model": model_name,
            "false_negatives": int(fn),
            "false_positives": int(fp),
            "weighted_penalty": weighted_penalty,
            "cost_weighted_score": (
                1 - weighted_penalty / worst_case_penalty if worst_case_penalty else np.nan
            ),
        }

    cost_comparison = pd.DataFrame(
        [
            cost_weighted_summary(comparison["actual_churn"], comparison["baseline_predicted"], "Baseline"),
            cost_weighted_summary(comparison["actual_churn"], comparison["llm_predicted"], "LLM"),
        ]
    )
    print("Cost-sensitive comparison (FN=5, FP=1; higher normalized score is better):")
    display(cost_comparison.style.format({"cost_weighted_score": "{:.4f}"}))
    print("Score formula: 1 - (5 × FN + FP) / (5 × actual churners + actual non-churners).")
else:
    print("Segment disagreement and cost-weighted analyses are pending the merged prediction table.")

Segment disagreement and cost-weighted analyses are pending the merged prediction table.
